In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
from statistics import mean
import sys
sys.path.append("..")  # helpers.py лежит в корне курса
from helpers import add_user_message, chat, print_results

client = Anthropic()
model = "claude-haiku-5-5"

In [2]:
def run_prompt(test_case):
    """Подставляет задачу в prompt и возвращает ответ Claude"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(client, model, messages)

In [3]:
# Structured outputs вместо prefill: API возвращает JSON по схеме
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "strengths": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                        "weaknesses": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                        "reasoning": {"type": "string"},
                        "score": {"type": "integer"},
                    },
                    "required": [
                        "strengths", "weaknesses", "reasoning", "score",
                    ],
                    "additionalProperties": False,
                },
            }
        },
    )
    text = next(
        block.text for block in response.content if block.type == "text"
    )
    return json.loads(text)

In [4]:
def run_test_case(test_case):
    """Вызывает run_prompt и оценивает результат через grader"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def run_eval(dataset):
    """Запускает run_test_case для каждой задачи и печатает среднюю оценку"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

In [5]:
# dataset.json сохранили в уроке 11
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.333333333333334


In [6]:
for result in results:
    print("SCORE:", result["score"])
    print("REASONING:", result["reasoning"])
    print("-" * 80)

SCORE: 8
REASONING: The solution correctly solves the task. The paginator handles multi-page listings, the empty-page guard is appropriate, and the endswith filter is correct. The notes are helpful and cover case sensitivity, credentials, and prefix filtering. The main flaw is the docstring/implementation mismatch on the exception type, which an API consumer relying on the docstring would find misleading. Minor issues include the Python version requirement and the per-call client creation. These are small against an otherwise correct, idiomatic implementation.
--------------------------------------------------------------------------------
SCORE: 9
REASONING: The policy is functionally correct and minimal. It grants exactly s3:ListBucket on the bucket ARN and s3:GetObject on the object ARNs, uses valid syntax, and the explanation accurately describes IAM's resource-level distinction. Minor deductions for not noting KMS requirements that commonly break read access in production buckets 

In [7]:
print_results(results)

TASK: Write a Python function using boto3 that takes an S3 bucket name and returns a list of all object keys in the bucket that end with '.csv', handling pagination correctly.
SCORE: 8
Here's a Python function that uses boto3 to list all object keys ending with `.csv` in a given S3 bucket, using a paginator so it handles buckets with more than 1,000 objects correctly:

```python
import boto3
from botocore.exceptions import ClientError


def list_csv_keys(bucket_name: str) -> list[str]:
    """
    Return a list of all object keys in the given S3 bucket that end with '.csv'.

    Uses a paginator so that results beyond the 1,000-object limit of a single
    ListObjectsV2 call are retrieved correctly.

    Args:
        bucket_name: The name of the S3 bucket.

    Returns:
        A list of object keys ending with '.csv'.

    Raises:
        botocore.exceptions.ClientError: If the bucket cannot be accessed
            (e.g., it does not exist or permissions are insufficient).
    """
  